In [0]:
spark.conf.set("spark.databricks.cloudFiles.schemaInference.sampleSize.numBytes", "1gb")

In [0]:
catalog = "stuart"
schema = "lv"

In [0]:
raw_data_path = f"/Volumes/{catalog}/{schema}/csv"

raw_df = (
  spark.readStream
  .format("cloudfiles")
  .option("cloudFiles.format", "csv")
  .option("cloudFiles.schemaLocation", f"{raw_data_path}/_schema")
  .option("cloudFiles.inferColumnTypes", "true")
  .load(raw_data_path)
  )

In [0]:
raw_tref = f"{catalog}.{schema}.ais_raw"

In [0]:
(
  raw_df
  .writeStream
  .option("checkpointLocation", f"{raw_data_path}/_checkpoints")
  .trigger(once=True)
  .toTable(raw_tref, outputMode="append")
  )

In [0]:
print(f"{spark.table(raw_tref).count()=:,}")

In [0]:
spark.table(raw_tref).display()

In [0]:
import pyspark.sql.functions as F
import pyspark.databricks.sql.functions as DBF

In [0]:
h3_resolution = 8

In [0]:
bronze_df = (
  spark.table(raw_tref)
  .withColumn("point_geom", DBF.st_point("LON", "LAT", 4326))
  .withColumn(f"h3_r{h3_resolution}", DBF.h3_longlatash3("LON", "LAT", h3_resolution))
)

In [0]:
bronze_tref = f"{catalog}.{schema}.ais_bronze"

In [0]:
(
  bronze_df
  .write
  .mode("overwrite")
  .saveAsTable(bronze_tref)
)

In [0]:
print(f"{spark.table(bronze_tref).count()=:,}")

In [0]:
spark.table(bronze_tref).display()